## **Proceso ETL**

### **1. Importación de librerías**

In [167]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import time

### **2. Configuración de la conexión**

In [168]:
USUARIO = "postgres"
PASSWORD = "1028"
HOST = "localhost"
PUERTO = "5432"
BASE_DATOS = "dw_analisis_criminalidad"

engine = create_engine(
    f"postgresql+psycopg2://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{BASE_DATOS}"
)

print("Conexión creada correctamente.")

Conexión creada correctamente.


Prueba de conexión

In [169]:
query = "SELECT * FROM dim_delito"

df = pd.read_sql(query, engine)
df.head()

,id_delito,tipo_delito
0,1,Amenazas
1,2,Delitos sexuales
2,3,Homicidio
3,4,Hurto a residencias y entidades comerciales
4,5,Hurto de motocicletas y automotores


### **3. Extracción de datos**

Nombre para columna tipo delito de acuerdo con nombre de archivo

In [170]:
TIPOS_DELITO = {
    "Reporte amenazas.csv": "Amenazas",
    "Reporte delitos sexuales.csv": "Delitos sexuales",
    "Reporte homicidios.csv": "Homicidio",
    "Reporte hurto motocicletas y automotores.csv": "Hurto de motocicletas y automotores",
    "Reporte hurto residencias y entidades comerciales.csv": "Hurto a residencias y entidades comerciales",
    "Reporte lesiones personales y accidente de transito.csv": "Lesiones personales",
    "Reporte violencia intrafamiliar.csv": "Violencia intrafamiliar"
}

Leer y transformar archivos

In [171]:
# Ruta de los archivos CSV
RUTA_DATOS_CSV = Path("Datos CSV")

In [172]:
archivos = sorted(RUTA_DATOS_CSV.glob("*.csv"))

print(f"Se encontraron {len(archivos)} archivos:\n")

for archivo in archivos:
    print(f"- {archivo.name}")

Se encontraron 7 archivos:

- Reporte amenazas.csv
- Reporte delitos sexuales.csv
- Reporte homicidios.csv
- Reporte hurto motocicletas y automotores.csv
- Reporte hurto residencias y entidades comerciales.csv
- Reporte lesiones personales y accidente de transito.csv
- Reporte violencia intrafamiliar.csv


In [173]:
dataframes = []

for archivo in archivos:

    print(f"Leyendo {archivo.name}...")

    df = pd.read_csv(
        archivo,
        encoding="utf-8-sig",
        low_memory=False
    )

    # Estandarizar nombres de columnas
    df.columns = (
        df.columns
          .str.strip()
          .str.upper()
    )

    # Corregir diferencias del archivo de homicidios
    if "GRUPO EDAD" in df.columns:

        df.rename(
            columns={
                "GRUPO EDAD": "GRUPO ETARIO"
            },
            inplace=True
        )

    # Eliminar columnas que no se utilizarán
    df.drop(
        columns=[
            "CODIGO DANE",
            "DELITOS",
            "DELITO",
            "TIPO DE HURTO",
            "DESCRIPCIÓN CONDUCTA"
        ],
        errors="ignore",
        inplace=True
    )

    # Crear nuestra columna DELITO
    df["DELITO"] = TIPOS_DELITO[archivo.name]

    dataframes.append(df)

    print(f"   Registros: {len(df):,}")

Leyendo Reporte amenazas.csv...
   Registros: 650,347
Leyendo Reporte delitos sexuales.csv...
   Registros: 392,576
Leyendo Reporte homicidios.csv...
   Registros: 154,428
Leyendo Reporte hurto motocicletas y automotores.csv...
   Registros: 641,663
Leyendo Reporte hurto residencias y entidades comerciales.csv...
   Registros: 633,804
Leyendo Reporte lesiones personales y accidente de transito.csv...
   Registros: 1,351,421
Leyendo Reporte violencia intrafamiliar.csv...
   Registros: 682,558


Unificar todos los dataframes

In [174]:
df = pd.concat(
    dataframes,
    ignore_index=True
)

Verificar resultado

In [175]:
print(f"\nTotal de registros: {len(df):,}")

print("\nColumnas:")

print(df.columns.tolist())


Total de registros: 4,506,797

Columnas:
['DEPARTAMENTO', 'MUNICIPIO', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD', 'DELITO']


Visualizar primeros registros

In [176]:
df.head()

,DEPARTAMENTO,MUNICIPIO,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD,DELITO
0,SUCRE,Coveñas,NO REPORTADO,22/06/2013,MASCULINO,ADULTOS,1,Amenazas
1,ATLÁNTICO,Barranquilla (CT),NO REPORTADO,13/12/2013,MASCULINO,ADULTOS,1,Amenazas
2,CAQUETÁ,Florencia (CT),NO REPORTADO,12/06/2012,FEMENINO,ADULTOS,1,Amenazas
3,VALLE,Cartago,NO REPORTADO,01/04/2012,MASCULINO,ADULTOS,1,Amenazas
4,VALLE,Cali (CT),NO REPORTADO,20/09/2012,FEMENINO,ADULTOS,1,Amenazas


### **4. Transformación**

Se define el periodo de estudio 

In [177]:
ANIO_INICIO = 2014
ANIO_FIN = 2024

Informacion general del dataframe

In [178]:
print(f"Registros: {len(df):,}")
print(f"Columnas: {len(df.columns)}")

Registros: 4,506,797
Columnas: 8


In [179]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4506797 entries, 0 to 4506796
Data columns (total 8 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   DEPARTAMENTO  object
 1   MUNICIPIO     object
 2   ARMAS MEDIOS  object
 3   FECHA HECHO   object
 4   GENERO        object
 5   GRUPO ETARIO  object
 6   CANTIDAD      int64 
 7   DELITO        object
dtypes: int64(1), object(7)
memory usage: 275.1+ MB


In [180]:
df.describe(include="all")

,DEPARTAMENTO,MUNICIPIO,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD,DELITO
count,4506797,4506769,4506797,4506797,3926133,4447274,4.506797e+06,4506797
unique,33,1023,53,5934,3,3,NaN,7
top,BOGOTA,Bogotá D.C. (CT),SIN EMPLEO DE ARMAS,01/01/2017,MASCULINO,ADULTOS,NaN,Lesiones personales
freq,567222,581051,1775380,1991,2116257,4080236,NaN,1351421
mean,NaN,NaN,NaN,NaN,NaN,NaN,1.377545e+00,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,2.420378e+00,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN


#### **4.1. Renombrar columnas**

In [181]:
df.columns = [
    "departamento",
    "municipio",
    "armas_medios",
    "fecha_hecho",
    "genero",
    "grupo_etario",
    "cantidad",
    "delito"
]

#### **4.2. Convertir tipos de datos**

In [182]:
df["fecha_hecho"] = pd.to_datetime(
    df["fecha_hecho"],
    errors="coerce"
)

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_10492\933075919.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["fecha_hecho"] = pd.to_datetime(


In [183]:
df["cantidad"] = pd.to_numeric(
    df["cantidad"],
    errors="coerce"
)

In [184]:
df.dtypes

departamento            object
municipio               object
armas_medios            object
fecha_hecho     datetime64[ns]
genero                  object
grupo_etario            object
cantidad                 int64
delito                  object
dtype: object

In [185]:
columnas_texto = [
    "departamento",
    "municipio",
    "armas_medios",
    "genero",
    "grupo_etario",
    "delito"
]

for columna in columnas_texto:

    df[columna] = (
        df[columna]
        .astype("string")
        .str.strip()
    )

In [186]:
df.head()

,departamento,municipio,armas_medios,fecha_hecho,genero,grupo_etario,cantidad,delito
0,SUCRE,Coveñas,NO REPORTADO,2013-06-22,MASCULINO,ADULTOS,1,Amenazas
1,ATLÁNTICO,Barranquilla (CT),NO REPORTADO,2013-12-13,MASCULINO,ADULTOS,1,Amenazas
2,CAQUETÁ,Florencia (CT),NO REPORTADO,2012-06-12,FEMENINO,ADULTOS,1,Amenazas
3,VALLE,Cartago,NO REPORTADO,2012-04-01,MASCULINO,ADULTOS,1,Amenazas
4,VALLE,Cali (CT),NO REPORTADO,2012-09-20,FEMENINO,ADULTOS,1,Amenazas


#### **4.3. Revision de columnas y manejo de nulos**

##### **Genero**

In [187]:
df["genero"].value_counts(dropna=False)

genero
MASCULINO     2116257
FEMENINO      1807477
<NA>           580664
NO REPORTA       2399
Name: count, dtype: Int64

En este caso, un NA se puede relacionar con un no reportado

In [188]:
# Reemplazar valores faltantes y "NO REPORTA"

df["genero"] = (
    df["genero"]
        .fillna("NO REPORTADO")
        .replace({
            "NO REPORTA": "NO REPORTADO"
        })
)

In [189]:
df["genero"] = df["genero"].replace({
    "MASCULINO": "Masculino",
    "FEMENINO": "Femenino",
    "NO REPORTADO": "No reportado"
})

Se estandariza el texto con mayuscula inicial

In [190]:
df["genero"].value_counts(dropna=False)

genero
Masculino       2116257
Femenino        1807477
No reportado     583063
Name: count, dtype: Int64

##### **Grupo etario**

In [191]:
df["grupo_etario"].value_counts(dropna=False)

grupo_etario
ADULTOS         4080236
ADOLESCENTES     224688
MENORES          142350
<NA>              59523
Name: count, dtype: Int64

Los NA se tratan como no reportado y se modifica a mayuscula inicial

In [192]:
# Estandarizar grupo etario
df["grupo_etario"] = (
    df["grupo_etario"]
        .fillna("NO REPORTADO")
        .replace({
            "ADULTOS": "Adultos",
            "ADOLESCENTES": "Adolescentes",
            "MENORES": "Menores",
            "NO REPORTADO": "No reportado"
        })
)

In [193]:
df["grupo_etario"].value_counts(dropna=False)

grupo_etario
Adultos         4080236
Adolescentes     224688
Menores          142350
No reportado      59523
Name: count, dtype: Int64

In [194]:
df["armas_medios"].value_counts(dropna=False)

armas_medios
SIN EMPLEO DE ARMAS                   1775380
CONTUNDENTES                           919974
ARMA DE FUEGO                          480933
ARMA BLANCA / CORTOPUNZANTE            408852
NO REPORTADO                           287172
LLAVE MAESTRA                          228429
VEHICULO                               185133
MOTO                                   123168
PALANCAS                                39441
ESCOPOLAMINA                            16357
ACIDO                                   13286
PUNZANTES                                5754
CORTANTES                                3875
BICICLETA                                2812
ARMA TRAUMATICA                          2300
PERRO                                    2298
ARTEFACTO EXPLOSIVO/CARGA DINAMITA       1766
CORTOPUNZANTES                           1465
LICOR ADULTERADO                         1012
COMBUSTIBLE                               940
ALIMENTOS VENCIDOS                        745
GRANADA DE MANO      

In [195]:
# Estandarizar categorías de armas o medios

df["armas_medios"] = (
    df["armas_medios"]
        .fillna("No reportado")
        .replace({
            "NO REPORTADO": "No reportado",

            "ARMAS BLANCAS": "ARMA BLANCA / CORTOPUNZANTE",
            "CORTOPUNZANTES": "ARMA BLANCA / CORTOPUNZANTE",
            "CUCHILLA": "ARMA BLANCA / CORTOPUNZANTE",

            "POLVORA(FUEGOS PIROTECNICOS)": "POLVORA (FUEGOS PIROTECNICOS)"
        })
)

##### **Departamento**

In [196]:
df["departamento"].nunique()

33

In [197]:
df["departamento"].sort_values().unique()

<StringArray>
[          'AMAZONAS',          'ANTIOQUIA',             'ARAUCA',
          'ATLÁNTICO',             'BOGOTA',            'BOLÍVAR',
             'BOYACÁ',             'CALDAS',            'CAQUETÁ',
           'CASANARE',              'CAUCA',              'CESAR',
              'CHOCÓ',       'CUNDINAMARCA',            'CÓRDOBA',
            'GUAINÍA',            'GUAJIRA',           'GUAVIARE',
              'HUILA',          'MAGDALENA',               'META',
             'NARIÑO', 'NORTE DE SANTANDER',           'PUTUMAYO',
            'QUINDÍO',          'RISARALDA',         'SAN ANDRÉS',
          'SANTANDER',              'SUCRE',             'TOLIMA',
              'VALLE',             'VAUPÉS',            'VICHADA']
Length: 33, dtype: string

Se escribe como titulo y se modifican nombres oficiales

In [198]:
# Cambiar a formato de presentación
df["departamento"] = df["departamento"].str.title()

# Correcciones de nombres oficiales
df["departamento"] = df["departamento"].replace({
    "Bogota": "Bogotá D.C.",
    "Guajira": "La Guajira",
    "Valle": "Valle del Cauca"
})

##### **Municipio**

In [199]:
print(f"Municipios únicos: {df['municipio'].nunique():,}")
print(f"Registros nulos: {df['municipio'].isna().sum():,}")

Municipios únicos: 1,023
Registros nulos: 28


Se cambian nulos por "No reportado"

In [200]:
df["municipio"] = df["municipio"].fillna("No reportado")

In [201]:
df["municipio"] = (
    df["municipio"].str.title()
)

In [202]:
df["municipio"] = (
    df["municipio"]
        .str.replace(" Del ", " del ", regex=False)
        .str.replace(" De ", " de ", regex=False)
        .str.replace(" La ", " la ", regex=False)
        .str.replace(" Las ", " las ", regex=False)
        .str.replace(" Los ", " los ", regex=False)
)

In [203]:
municipios = sorted(df["municipio"].dropna().unique())

print(f"Total de municipios: {len(municipios)}")
pd.Series(municipios).to_csv(
    "Salidas/municipios_unicos.csv",
    index=False,
    header=["municipio"],
    encoding="utf-8-sig"
)

Total de municipios: 1024


##### **Fecha**

In [204]:
print(df["fecha_hecho"].isna().sum())

print(df["fecha_hecho"].min())
print(df["fecha_hecho"].max())

0
2010-01-01 00:00:00
2026-03-31 00:00:00


Delimitacion por periodo de estudio

In [205]:
df.shape

(4506797, 8)

In [206]:
# Filtrar únicamente el periodo de estudio
df = df[
    (df["fecha_hecho"].dt.year >= ANIO_INICIO) &
    (df["fecha_hecho"].dt.year <= ANIO_FIN)
].copy()

In [207]:
print(df["fecha_hecho"].min())
print(df["fecha_hecho"].max())

2014-01-01 00:00:00
2024-12-31 00:00:00


In [208]:
print(f"Registros para el estudio: {len(df):,}")

Registros para el estudio: 3,481,420


##### **Validacion de nulos**

In [209]:
# Conteo de valores nulos
nulos = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .rename("Valores nulos")
)

print(nulos)

departamento    0
municipio       0
armas_medios    0
fecha_hecho     0
genero          0
grupo_etario    0
cantidad        0
delito          0
Name: Valores nulos, dtype: int64


#### **4.5. Consolidación de registros**

Con el fin de garantizar una granularidad consistente en la tabla de hechos del Data Warehouse, los registros se consolidan agrupándolos por todas las dimensiones definidas (tiempo, ubicación, delito, arma o medio y características de la víctima). Para cada combinación única de dimensiones, la medida **cantidad** se obtiene mediante la suma de los valores correspondientes. Esta transformación reduce la redundancia en los datos sin alterar el número total de delitos reportados por la fuente.

In [210]:
# Definir las dimensiones de la tabla de hechos
columnas_dimension = [
    "fecha_hecho",
    "departamento",
    "municipio",
    "delito",
    "armas_medios",
    "genero",
    "grupo_etario"
]

# Consolidar registros con la misma combinación de dimensiones
df = (
    df.groupby(columnas_dimension, as_index=False)
      .agg(cantidad=("cantidad", "sum"))
)

# Verificar el resultado
print(f"Registros después de la consolidación: {len(df):,}")
print(f"Total de delitos: {df['cantidad'].sum():,}")

df.head()

Registros después de la consolidación: 2,786,613
Total de delitos: 4,805,136


,fecha_hecho,departamento,municipio,delito,armas_medios,genero,grupo_etario,cantidad
0,2014-01-01,Antioquia,Amagá,Amenazas,SIN EMPLEO DE ARMAS,Femenino,Adultos,1
1,2014-01-01,Antioquia,Amagá,Lesiones personales,ARMA BLANCA / CORTOPUNZANTE,Masculino,Adultos,1
2,2014-01-01,Antioquia,Andes,Lesiones personales,CONTUNDENTES,Femenino,Adultos,3
3,2014-01-01,Antioquia,Angostura,Lesiones personales,CONTUNDENTES,Masculino,Adultos,1
4,2014-01-01,Antioquia,Angostura,Lesiones personales,VEHICULO,Femenino,Adultos,1


In [211]:
duplicados = df.duplicated(subset=columnas_dimension).sum()

print(f"Duplicados después de la consolidación: {duplicados:,}")

Duplicados después de la consolidación: 0


### **5. Carga de dimensiones**

##### **5.1. Construcción de la dimensión Tiempo**

La dimensión de tiempo se construye a partir de las fechas únicas presentes en el conjunto de datos. Para cada fecha se generan los atributos temporales que facilitarán el análisis de la información desde diferentes perspectivas cronológicas, incluyendo día, mes, año, trimestre y la identificación de fines de semana.

In [212]:
# Construir la dimensión tiempo
dim_tiempo = (
    df[["fecha_hecho"]]
    .drop_duplicates()
    .sort_values("fecha_hecho")
    .reset_index(drop=True)
)

# Diccionarios para nombres
dias = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo"
}

meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

# Atributos
dim_tiempo["dia"] = dim_tiempo["fecha_hecho"].dt.day
dim_tiempo["dia_semana"] = dim_tiempo["fecha_hecho"].dt.weekday + 1
dim_tiempo["nombre_dia"] = dim_tiempo["fecha_hecho"].dt.weekday.map(dias)
dim_tiempo["es_fin_semana"] = dim_tiempo["fecha_hecho"].dt.weekday >= 5

dim_tiempo["mes"] = dim_tiempo["fecha_hecho"].dt.month
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(meses)

dim_tiempo["trimestre"] = dim_tiempo["fecha_hecho"].dt.quarter
dim_tiempo["anio"] = dim_tiempo["fecha_hecho"].dt.year

dim_tiempo.head()

,fecha_hecho,dia,dia_semana,nombre_dia,es_fin_semana,mes,nombre_mes,trimestre,anio
0,2014-01-01,1,3,Miércoles,False,1,Enero,1,2014
1,2014-01-02,2,4,Jueves,False,1,Enero,1,2014
2,2014-01-03,3,5,Viernes,False,1,Enero,1,2014
3,2014-01-04,4,6,Sábado,True,1,Enero,1,2014
4,2014-01-05,5,7,Domingo,True,1,Enero,1,2014


In [213]:
dim_tiempo.shape

(4018, 9)

##### **5.2. Construcción de la dimensión Ubicación**

La dimensión de ubicación se construye a partir de las combinaciones únicas de departamento y municipio presentes en los registros. Esta dimensión permite analizar la ocurrencia de los delitos desde una perspectiva geográfica, facilitando consultas tanto a nivel departamental como municipal.

In [214]:
# Construir la dimensión ubicación

dim_ubicacion = (
    df[["departamento", "municipio"]]
    .drop_duplicates()
    .sort_values(["departamento", "municipio"])
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_ubicacion):,}")

dim_ubicacion.head()

Registros: 1,107


,departamento,municipio
0,Amazonas,Leticia (Ct)
1,Amazonas,Puerto Nariño
2,Antioquia,Abejorral
3,Antioquia,Abriaquí
4,Antioquia,Alejandría


##### **5.3. Construcción de la dimensión Delito**

La dimensión de delito se construye a partir de los tipos de delito presentes en el conjunto de datos. Su propósito es clasificar los registros de la tabla de hechos según la naturaleza del delito, permitiendo realizar análisis comparativos entre las diferentes categorías consideradas en la investigación.

In [215]:
# Construir la dimensión delito

dim_delito = (
    df[["delito"]]
    .drop_duplicates()
    .sort_values("delito")
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_delito)}")

dim_delito

Registros: 7


,delito
0,Amenazas
1,Delitos sexuales
2,Homicidio
3,Hurto a residencias y entidades comerciales
4,Hurto de motocicletas y automotores
5,Lesiones personales
6,Violencia intrafamiliar


##### **5.4. Construcción de la dimensión Arma**

La dimensión de arma se construye a partir de los valores únicos del atributo armas o medios. Esta dimensión permite analizar la distribución de los delitos según el arma o medio utilizado durante la ocurrencia del hecho.

In [216]:
# Construir la dimensión arma

dim_arma = (
    df[["armas_medios"]]
    .drop_duplicates()
    .sort_values("armas_medios")
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_arma)}")

dim_arma.head(10)

Registros: 49


,armas_medios
0,ACIDO
1,AGUA CALIENTE
2,ALIMENTOS VENCIDOS
3,ALMOHADA
4,ALUCINOGENOS
5,ARMA BLANCA / CORTOPUNZANTE
6,ARMA DE FUEGO
7,ARMA TRAUMATICA
8,ARTEFACTO EXPLOSIVO/CARGA DINAMITA
9,ARTEFACTO INCENDIARIO


##### **5.5. Construcción de la dimensión Víctima**

La dimensión de víctima se construye a partir de las combinaciones únicas de género y grupo etario presentes en los registros. Esta dimensión permite caracterizar a las víctimas de los hechos delictivos y realizar análisis según sus características demográficas.

In [217]:
# Construir la dimensión víctima

dim_victima = (
    df[["genero", "grupo_etario"]]
    .drop_duplicates()
    .sort_values(["genero", "grupo_etario"])
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_victima)}")

dim_victima

Registros: 10


,genero,grupo_etario
0,Femenino,Adolescentes
1,Femenino,Adultos
2,Femenino,Menores
3,Masculino,Adolescentes
4,Masculino,Adultos
5,Masculino,Menores
6,No reportado,Adolescentes
7,No reportado,Adultos
8,No reportado,Menores
9,No reportado,No reportado


##### **5.6. Validacion y carga**

In [218]:
print("Dim_Tiempo:", len(dim_tiempo))
print("Dim_Ubicacion:", len(dim_ubicacion))
print("Dim_Delito:", len(dim_delito))
print("Dim_Arma:", len(dim_arma))
print("Dim_Victima:", len(dim_victima))

Dim_Tiempo: 4018
Dim_Ubicacion: 1107
Dim_Delito: 7
Dim_Arma: 49
Dim_Victima: 10


In [219]:
print(dim_tiempo.duplicated().sum())
print(dim_ubicacion.duplicated().sum())
print(dim_delito.duplicated().sum())
print(dim_arma.duplicated().sum())
print(dim_victima.duplicated().sum())

0
0
0
0
0


Borrar contenido de tablas

In [220]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE fact_delitos RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_tiempo RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_ubicacion RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_delito RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_arma RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_victima RESTART IDENTITY CASCADE;"))

Homologaciones necesarias

In [221]:
# Dimensión Tiempo
dim_tiempo = dim_tiempo.rename(
    columns={
        "fecha_hecho": "fecha"
    }
)

# Dimensión Delito
dim_delito = dim_delito.rename(
    columns={
        "delito": "tipo_delito"
    }
)

# Dimensión Arma
dim_arma = dim_arma.rename(
    columns={
        "armas_medios": "nombre_arma"
    }
)

Carga de dim_tiempo

In [222]:
dim_tiempo.to_sql(
    "dim_tiempo",
    engine,
    if_exists="append",
    index=False
)

18

Carga de dim_ubicacion

In [223]:
dim_ubicacion.to_sql(
    "dim_ubicacion",
    engine,
    if_exists="append",
    index=False
)

107

Carga de dim_delito

In [224]:
dim_delito.to_sql(
    "dim_delito",
    engine,
    if_exists="append",
    index=False
)

7

Carga de dim_arma

In [225]:
dim_arma.to_sql(
    "dim_arma",
    engine,
    if_exists="append",
    index=False
)

49

Carga de dim_victima

In [226]:
dim_victima.to_sql(
    "dim_victima",
    engine,
    if_exists="append",
    index=False
)

10

### **6. Carga de tabla de hechos Delitos**

#### **6.1. Recuperación de las dimensiones**

Una vez cargadas las dimensiones en PostgreSQL, se consultan nuevamente para recuperar las claves sustitutas generadas automáticamente. Estas claves serán utilizadas para establecer las relaciones entre las dimensiones y la tabla de hechos durante la construcción del modelo estrella.

In [227]:
# Leer las dimensiones desde PostgreSQL

dim_tiempo_dw = pd.read_sql(
    "SELECT * FROM dim_tiempo",
    engine,
    parse_dates=["fecha"]
)

dim_ubicacion_dw = pd.read_sql(
    "SELECT * FROM dim_ubicacion",
    engine
)

dim_delito_dw = pd.read_sql(
    "SELECT * FROM dim_delito",
    engine
)

dim_arma_dw = pd.read_sql(
    "SELECT * FROM dim_arma",
    engine
)

dim_victima_dw = pd.read_sql(
    "SELECT * FROM dim_victima",
    engine
)

In [232]:
print(dim_tiempo_dw.head())

print("\n", dim_ubicacion_dw.head())

print("\n", dim_delito_dw.head())

print("\n", dim_arma_dw.head())

print("\n", dim_victima_dw.head())

   id_tiempo      fecha  dia  mes nombre_mes  trimestre  anio  dia_semana  \
0          1 2014-01-01    1    1      Enero          1  2014           3   
1          2 2014-01-02    2    1      Enero          1  2014           4   
2          3 2014-01-03    3    1      Enero          1  2014           5   
3          4 2014-01-04    4    1      Enero          1  2014           6   
4          5 2014-01-05    5    1      Enero          1  2014           7   

  nombre_dia  es_fin_semana  
0  Miércoles          False  
1     Jueves          False  
2    Viernes          False  
3     Sábado           True  
4    Domingo           True  

    id_ubicacion departamento      municipio
0             1     Amazonas   Leticia (Ct)
1             2     Amazonas  Puerto Nariño
2             3    Antioquia      Abejorral
3             4    Antioquia       Abriaquí
4             5    Antioquia     Alejandría

    id_delito                                  tipo_delito
0          1                   

#### **6.2. Relacionar dataframe principal con dimensiones**

Merge con dim tiempo

In [246]:
fact_delitos = df.merge(
    dim_tiempo_dw[["id_tiempo", "fecha"]],
    left_on="fecha_hecho",
    right_on="fecha",
    how="left"
)

print(f"Registros: {len(fact_delitos):,}")

Registros: 2,786,613


In [247]:
print("id_tiempo nulos:", fact_delitos["id_tiempo"].isna().sum())

id_tiempo nulos: 0


Merge con dim ubicacion

In [248]:
fact_delitos = fact_delitos.merge(
    dim_ubicacion_dw,
    on=["departamento", "municipio"],
    how="left"
)

In [237]:
print("id_ubicacion nulos:", fact_delitos["id_ubicacion"].isna().sum())

id_ubicacion nulos: 0


Merge con dim delito

In [249]:
fact_delitos = fact_delitos.merge(
    dim_delito_dw,
    left_on="delito",
    right_on="tipo_delito",
    how="left"
)

In [239]:
print("id_delito nulos:", fact_delitos["id_delito"].isna().sum())

id_delito nulos: 0


Merge con dim arma

In [250]:
fact_delitos = fact_delitos.merge(
    dim_arma_dw,
    left_on="armas_medios",
    right_on="nombre_arma",
    how="left"
)

In [244]:
print("id_arma nulos:", fact_delitos["id_arma"].isna().sum())

id_arma nulos: 0


Merge con dim victima

In [252]:
fact_delitos = fact_delitos.merge(
    dim_victima_dw,
    on=["genero", "grupo_etario"],
    how="left"
)

In [253]:
print("id_victima nulos:", fact_delitos["id_victima"].isna().sum())

id_victima nulos: 0


#### **6.3. Validación y carga**

In [256]:
fact_delitos = fact_delitos[
    [
        "id_delito",
        "id_tiempo",
        "id_ubicacion",
        "id_victima",
        "id_arma",
        "cantidad"
    ]
].copy()

fact_delitos = fact_delitos.rename(
    columns={
        "cantidad": "numero"
    }
)

fact_delitos.head()

,id_delito,id_tiempo,id_ubicacion,id_victima,id_arma,numero
0,1,1,6,2,46,1
1,6,1,6,5,6,1
2,6,1,8,2,18,3
3,6,1,10,5,18,1
4,6,1,10,2,48,1


In [257]:
print(f"Registros de la tabla de hechos: {len(fact_delitos):,}")

print(f"Suma total de delitos: {fact_delitos['numero'].sum():,}")

Registros de la tabla de hechos: 2,786,613
Suma total de delitos: 4,805,136


In [258]:
print(fact_delitos.isna().sum())

id_delito       0
id_tiempo       0
id_ubicacion    0
id_victima      0
id_arma         0
numero          0
dtype: int64


In [260]:
fact_delitos = fact_delitos.rename(
    columns={
        "numero": "cantidad_delitos"
    }
)

fact_delitos.to_sql(
    "fact_delitos",
    engine,
    if_exists="append",
    index=False
)

print(f"Fact_Delitos cargada correctamente ({len(fact_delitos):,} registros).")

Fact_Delitos cargada correctamente (2,786,613 registros).
